In [3]:
import torch
import pandas as pd
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForTokenClassification

In [4]:
# ─── 1) Your tag2idx and inverse mapping ──────────────────────────────────────
tag2idx = {
    'I-COMPANY': 0,  'U-COMPANY': 1,   'I-EMAIL': 2,     'I-GRADYEAR': 3,
    'L-EMAIL': 4,    '[SEP]': 5,       'B-CLG': 6,       'U-SKILLS': 7,
    'X': 8,          'L-NAME': 9,      'B-DEG': 10,      'U-EMAIL': 11,
    'L-CLG': 12,     'L-GRADYEAR': 13, 'L-DESIG': 14,    'U-DEG': 15,
    'B-SKILLS': 16,  'L-LOC': 17,      'I-SKILLS': 18,   'U-GRADYEAR': 19,
    'B-NAME': 20,    'I-YOE': 21,      'B-YOE': 22,      'L-YOE': 23,
    'I-DEG': 24,     'B-COMPANY': 25,  'I-NAME': 26,     'I-DESIG': 27,
    'U-LOC': 28,     'B-GRADYEAR': 29, 'O': 30,          'U-YOE': 31,
    'I-CLG': 32,     'L-SKILLS': 33,   'B-LOC': 34,      'L-DEG': 35,
    'U-CLG': 36,     'U-DESIG': 37,    'B-EMAIL': 38,    'B-DESIG': 39,
    '[CLS]': 40,     'I-LOC': 41,      'L-COMPANY': 42
}
idx2tag = {v: k for k, v in tag2idx.items()}

In [5]:
# ─── 2) Device, tokenizer & model load ────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = BertTokenizer.from_pretrained("bert-base-cased")
model = BertForTokenClassification.from_pretrained(
    "bert-base-cased",
    num_labels=len(tag2idx)  # =43
)
state = torch.load("fifth_epoch.pt", map_location=device)
model.load_state_dict(state)
model.to(device)
model.eval()

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [6]:
# ─── 3) Dataset + DataLoader ─────────────────────────────────────────────────
class ResumeDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=128):
        enc = tokenizer(
            texts,
            truncation=True,
            padding="max_length",
            max_length=max_len,
            return_tensors="pt"
        )
        self.input_ids = enc["input_ids"]
        self.attention_mask = enc["attention_mask"]
    def __len__(self):
        return len(self.input_ids)
    def __getitem__(self, idx):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx]
        }

In [7]:
# load test data
df = pd.read_json("test.json")
texts = df["content"].tolist()
dataset = ResumeDataset(texts, tokenizer, max_len=128)
loader = DataLoader(dataset, batch_size=16)

In [8]:
# ─── 4) Inference ─────────────────────────────────────────────────────────────
all_label_sequences = []
with torch.no_grad():
    for batch in tqdm(loader, desc="Testing"):
        input_ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        outputs = model(input_ids=input_ids, attention_mask=mask)
        logits = outputs.logits                   # (batch, seq_len, 43)
        preds = torch.argmax(logits, dim=2)       # (batch, seq_len)
        # convert each token’s predicted id → tag name
        for pred_ids, attn in zip(preds.cpu().tolist(), mask.cpu().tolist()):
            seq_len = sum(attn)                  # number of real tokens (including special)
            tags = [idx2tag[i] for i in pred_ids[:seq_len]]
            all_label_sequences.append(tags)

Testing:   0%|                                                                                   | 0/7 [00:00<?, ?it/s]


RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [2]:
# ─── 5) Peek and/or save ─────────────────────────────────────────────────────
# Example: print tokens + their predicted tags for the first resume
first_ids = dataset.input_ids[0].tolist()
first_tokens = tokenizer.convert_ids_to_tokens(first_ids)
for tok, tag in zip(first_tokens, all_label_sequences[0]):
    print(f"{tok:12}: {tag}")
# Save results into a JSON file alongside original text
import json
out = [
    {"text": txt, "predicted_tags": tags}
    for txt, tags in zip(texts, all_label_sequences)
]
with open("test_predictions.json", "w") as f:
    json.dump(out, f, indent=2)
print("✅ Done! Predictions written to test_predictions.json")

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
